# Multi-Scale SR — Google Colab GPU Training

Pull the `multiscale_sr` code from GitHub, read the CMS jet parquet data from **Google Drive** (or an uploaded folder), train a chosen scale on the **full dataset** on the Colab GPU, evaluate tagging efficiency, and save everything under `/content` (mirrored to Drive) for download.

## Before you Run-All (prerequisites)

- **(P1) Push your latest code to GitHub first.** This notebook pulls code from git — any uncommitted local work will *not* be here. Commit + push the branch you want, then set `REPO_BRANCH` in **Cell 2** to match. *(The dim-collapse fixes must be pushed or you will train the old code.)*
- **(P2) Put the `*.parquet` jet files somewhere Colab can read.** Easiest is Google Drive. Set `DATA_DIR` in **Cell 4** to that folder.
- **(P3) Runtime settings:** *Runtime → Change runtime type → Hardware accelerator = GPU* (T4 is fine). Colab needs internet for `git clone` + `pip` (on by default).
- **(P4) Optional W&B:** add `WANDB_API_KEY` via the **🔑 Secrets** panel (left sidebar) with notebook access enabled, or paste it in **Cell 5**. No key → training runs with `--no-wandb`.

## How to run
1. Set `DATA_DIR` in **Cell 4**.
2. Set `SCALE` and `EPOCHS` in **Cell 6**.
3. *Runtime → Run all*.
4. Download results at the end (the last cell zips the run dir and offers a download).

## Cell 1 — Environment check (GPU is required)

In [ ]:
import subprocess, sys
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU. Enable it: Runtime -> Change runtime type -> Hardware accelerator = GPU (T4), "
        "then Runtime -> Run all again."
    )
print("device:", torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

torch: 2.11.0+cu128
cuda available: True
device: NVIDIA A100-SXM4-80GB
Thu Aug 13 15:33:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             51W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        | 

## Cell 2 — Get the code (clone repo at a pinned branch)
Re-running is safe: the clone dir is removed first. Set `REPO_BRANCH` to whatever you pushed in (P1).

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/rajveer43/cms-superres-reconstruction.git"
REPO_BRANCH = "master"
CLONE_DIR = Path("/content/repo")
CODE_DIR = CLONE_DIR / "multiscale_sr"

# Always leave the repo before deleting it
os.chdir("/content")

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        REPO_BRANCH,
        REPO_URL,
        str(CLONE_DIR),
    ],
    check=True,
)

os.chdir(CODE_DIR)
print("Current directory:", os.getcwd())

Current directory: /content/repo/multiscale_sr


## Cell 3 — Dependencies (install only what's missing)
Colab ships torch/numpy/pyarrow/sklearn/matplotlib/pyyaml. We only add `wandb` and `python-dotenv`; torch is **not** touched.

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
        print(f"ok: {pkg}")
    except ImportError:
        print(f"installing: {pkg}")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, imp in [("wandb", "wandb"), ("python-dotenv", "dotenv")]:
    ensure(pkg, imp)

import torch, numpy, pyarrow, sklearn
print("torch", torch.__version__, "| numpy", numpy.__version__,
      "| pyarrow", pyarrow.__version__, "| sklearn", sklearn.__version__)

ok: wandb
ok: python-dotenv
torch 2.11.0+cu128 | numpy 2.0.2 | pyarrow 18.1.0 | sklearn 1.6.1


## Cell 4 — Locate the data (Google Drive)
Mounts your Google Drive and points `DATA_DIR` at the folder holding the `*.parquet` files. **Edit `DATA_DIR`** to your actual path, then run. If you'd rather upload the files directly instead of using Drive, skip the mount and set `DATA_DIR` to wherever you put them (e.g. `/content/datasets`).

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

# --- Mount Google Drive (comment out if you uploaded data to /content instead) ---
from google.colab import drive
drive.mount("/content/drive")

# ================== EDIT THIS ONE LINE ==================
DATA_DIR = Path("/content/drive/MyDrive/GSoC_2026/DATASETS/QUARK_GLUON")   # <- folder that CONTAINS the *.parquet files
# =======================================================

files = sorted(DATA_DIR.glob('*.parquet'))
if not files:
    raise SystemExit(
        f"No *.parquet found in {DATA_DIR}. Fix DATA_DIR above to the folder that holds the "
        f"jet parquet files (list what is there with: !ls \"{DATA_DIR}\")."
    )

total = 0
print("DATA_DIR:", DATA_DIR)
for f in files:
    n = pq.ParquetFile(f).metadata.num_rows
    total += n
    print(f"  {f.name}: {n:,} rows")
print(f"TOTAL: {total:,} jets across {len(files)} file(s)")
print("(code splits by file: first files -> train, last -> val)")

Mounted at /content/drive
DATA_DIR: /content/drive/MyDrive/GSoC_2026/DATASETS/QUARK_GLUON
  QCDToGGQQ_IMGjet_RH1all_jet0_run0_n36272_LR.parquet: 36,272 rows
  QCDToGGQQ_IMGjet_RH1all_jet0_run1_n47540_LR.parquet: 47,540 rows
  QCDToGGQQ_IMGjet_RH1all_jet0_run2_n55494_LR.parquet: 55,494 rows
TOTAL: 139,306 jets across 3 file(s)
(code splits by file: first files -> train, last -> val)


## Cell 5 — W&B (optional)
Reads `WANDB_API_KEY` from the Colab **Secrets** panel (🔑, left sidebar) if present, else falls back to `--no-wandb`. You can also paste a key into the marked line. Never commit a key.

In [ ]:
import os

USE_WANDB = False

# Option A: Colab Secrets panel (recommended). Add a secret named WANDB_API_KEY and
# toggle notebook access on.
try:
    from google.colab import userdata
    _key = userdata.get("WANDB_API_KEY")
    if _key:
        os.environ["WANDB_API_KEY"] = _key
        USE_WANDB = True
        print("W&B: key found in Colab Secrets -> logging ENABLED")
except Exception as e:
    print("W&B: no Colab secret ->", type(e).__name__)

# Option B: paste a key here instead (leave empty to skip).
if not USE_WANDB:
    _PASTED_KEY = ""   # <- optionally paste your wandb key
    if _PASTED_KEY:
        os.environ["WANDB_API_KEY"] = _PASTED_KEY
        USE_WANDB = True
        print("W&B: using pasted key -> logging ENABLED")

if not USE_WANDB:
    print("W&B: DISABLED (training will use --no-wandb).")

W&B: key found in Colab Secrets -> logging ENABLED


## Cell 6 — Parameters (edit these)
Full dataset per epoch (no batch cap). On a T4, an ~84k-sample epoch is a few minutes, so a 20–30 epoch run fits comfortably. Results are written under `/content/experiments` and also copied to Drive at the end so they survive the session ending.

In [ ]:
SCALE            = 64                 # 16 / 32 / 64 (change this for different scale training)
EPOCHS           = 40                 # (change this for different epoch training)
# RUN_NAME is now dynamically generated in Cell 7 for multi-seed runs.
EXPERIMENTS_ROOT = "/content/experiments"
SEED             = 42                 # This is overridden in Cell 7 for multi-seed runs.
RESUME           = None               # e.g. '.../checkpoints/latest.pt' to continue

CONFIG = f"configs/scale_{SCALE}.yaml"
assert Path(CONFIG).exists(), f"missing {CONFIG} in {os.getcwd()} — check the clone (Cell 2)"
print(f"Base parameters for multi-seed runs (used in run name generation):")
print(f"scale={SCALE}  epochs={EPOCHS}  wandb={USE_WANDB}")
print(f"config={CONFIG}  data={DATA_DIR}  experiments_root={EXPERIMENTS_ROOT}")

Base parameters for multi-seed runs (used in run name generation):
scale=64  epochs=40  wandb=True
config=configs/scale_64.yaml  data=/content/drive/MyDrive/GSoC_2026/DATASETS/QUARK_GLUON  experiments_root=/content/experiments


In [ ]:
import shutil
from pathlib import Path
LOCAL_DATA = Path("/content/datasets")
LOCAL_DATA.mkdir(exist_ok=True)
for f in DATA_DIR.glob("*.parquet"):
    dst = LOCAL_DATA / f.name
    if not dst.exists():
        print("copying", f.name)
        shutil.copy(f, dst)
DATA_DIR = LOCAL_DATA   # train from local disk, not Drive


copying QCDToGGQQ_IMGjet_RH1all_jet0_run0_n36272_LR.parquet
copying QCDToGGQQ_IMGjet_RH1all_jet0_run1_n47540_LR.parquet
copying QCDToGGQQ_IMGjet_RH1all_jet0_run2_n55494_LR.parquet


## Cell 7 — Multi-seed sweep: train + evaluate + SAVE TO DRIVE, one seed at a time

Each seed is trained, evaluated, and **copied to Drive before the next seed starts**, so a Colab disconnect costs at most the run in flight. Already-saved seeds are skipped, so re-running this cell resumes a broken sweep.

Two things are deliberately held fixed across all seeds: `EVAL_SEED` and a single frozen HR tagger. Only the training seed varies, so the spread you measure is the model's, not the evaluation's. Watch that **HR AUC is identical for every seed** — if it moves, a seed has leaked into the measurement.

In [ ]:
# import multiscale_sr.utils.env as _env
# _orig = _env.resolve_env
# def _patched():
#     c = _orig()
#     return c.__class__(**{**c.__dict__, "num_workers": 0,
#                           "persistent_workers": False, "prefetch_factor": None})
# _env.resolve_env = _patched
# print("forced num_workers=0")


In [ ]:
import json, subprocess, sys, shutil, time
from pathlib import Path
import os

# Seeds to sweep. Each one trains, evaluates, and is SAVED TO DRIVE before the
# next begins -- so a disconnect mid-sweep costs at most the run in flight.
SEEDS_TO_RUN = [123, 456, 789, 999]

# Where completed runs are persisted. Each seed lands in its own subdirectory.
DRIVE_RUNS_ROOT = Path("/content/drive/MyDrive/multiscale_sr_runs")

# The MEASUREMENT seed, held FIXED across every SR seed. This is the whole point:
# if the eval seed moved with the training seed, the tagger's train/test split and
# init would change too, and model variance could not be separated from evaluation
# noise. HR/LR AUC must come out identical for every seed below.
EVAL_SEED = 0

# SEMD (top-K pixel approximation) -- the geometry-aware metric added to answer
# "why do all the physics metrics look stable while tagging swings 10%?".
# Every metric currently logged is invariant under pixel permutation, so none of
# them can see misplaced substructure; SEMD can. Held FIXED across seeds so the
# per-seed values are comparable. See multiscale_sr/spectral.py.
SEMD_TOPK        = 128   # brightest pixels kept per image (paper benchmarks use N=125)
SEMD_OMEGA_R     = 1.0   # angular scale where SR/HR energy imbalance is deposited
SEMD_MAX_SAMPLES = 1000  # images per checkpoint (SEMD is O(K^2 log K) per image)

# One frozen HR tagger, reused by every seed's evaluation. Trained on the first
# seed's run and loaded thereafter, so the ruler is literally the same object.
TAGGER_DIR = DRIVE_RUNS_ROOT / "taggers"
TAGGER_CKPT = TAGGER_DIR / f"hr_tagger_{SCALE}x.pt"

os.environ["MULTISCALE_SR_NUM_WORKERS"] = "0"

DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)
TAGGER_DIR.mkdir(parents=True, exist_ok=True)

all_run_results = {}
saved_to_drive = {}


def _stream(cmd):
    """Run a subprocess, echoing output live. Returns the exit code."""
    print(" ".join(str(c) for c in cmd), "\n", flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    return proc.returncode


def _save_to_drive(run_dir: Path, seed: int) -> Path | None:
    """Copy one finished run to Drive. Writes to a .partial staging name first
    and renames on success, so an interrupted copy can never be mistaken for a
    complete one on a later session."""
    final = DRIVE_RUNS_ROOT / run_dir.name
    staging = DRIVE_RUNS_ROOT / (run_dir.name + ".partial")
    try:
        for leftover in (staging, final):
            if leftover.exists():
                shutil.rmtree(leftover)
        t0 = time.time()
        shutil.copytree(run_dir, staging)
        staging.rename(final)
        size_mb = sum(f.stat().st_size for f in final.rglob("*") if f.is_file()) / 1e6
        print(f"\n[drive] SEED {seed} saved -> {final}  ({size_mb:.0f} MB, {time.time()-t0:.0f}s)")
        return final
    except Exception as e:
        print(f"\n[drive] WARNING: could not save SEED {seed}: {type(e).__name__}: {e}")
        print("[drive] the run is still at", run_dir, "for this session only.")
        return None


for current_seed in SEEDS_TO_RUN:
    print(f"\n{'='*80}\nStarting run for SEED = {current_seed}\n{'='*80}")

    _current_run_name = f"colab_{SCALE}x_full_{EPOCHS}_seed_{current_seed}"

    # Skip seeds already completed and persisted (lets you resume a broken sweep
    # by simply re-running this cell).
    _already = sorted(DRIVE_RUNS_ROOT.glob(f"*{SCALE}x*{_current_run_name}"))
    if _already:
        print(f"[skip] SEED {current_seed} already saved at {_already[-1]} — re-running would "
              f"overwrite it. Delete that directory to force a redo.")
        saved_to_drive[current_seed] = _already[-1]
        continue

    print(f"scale={SCALE}  epochs={EPOCHS}  run={_current_run_name}  wandb={USE_WANDB}  "
          f"train_seed={current_seed}  eval_seed={EVAL_SEED} (fixed)")

    train_cmd = [
        sys.executable, "train.py",
        "--config", CONFIG,
        "--data-dir", str(DATA_DIR),
        "--scale", str(SCALE),
        "--epochs", str(EPOCHS),
        "--run-name", _current_run_name,
        "--experiments-root", EXPERIMENTS_ROOT,
        "--cache",
        "--cache-dir", "/content/.sr_cache",
        "--seed", str(current_seed),
    ]
    if RESUME:
        train_cmd += ["--resume", RESUME]
    if not USE_WANDB:
        train_cmd += ["--no-wandb"]

    print("TRAIN RUN:", end=" ")
    if _stream(train_cmd) != 0:
        print(f"\nERROR: Training failed for SEED {current_seed} — skipping to next seed.")
        continue

    run_dirs = sorted(Path(EXPERIMENTS_ROOT).glob(f"*{SCALE}x*{_current_run_name}*"),
                      key=lambda p: p.stat().st_mtime)
    if not run_dirs:
        print(f"ERROR: Could not find run directory for SEED {current_seed}")
        continue

    current_run_dir = run_dirs[-1]
    print("\nTRAINING RUN_DIR:", current_run_dir)

    try:
        last_metrics_line = [l for l in (current_run_dir / "metrics.jsonl").read_text().splitlines() if l.strip()][-1]
        m = json.loads(last_metrics_line)
        print(f"Training Epoch={m.get('epoch')}  val_l1={m.get('val_l1'):.4f}  "
              f"val_psnr={m.get('val_psnr_norm'):.2f}  peak={m.get('val_peak_ratio'):.3f}  "
              f"energy_response={m.get('val_energy_response'):.4f}")
        training_metrics = m
    except Exception as e:
        print(f"WARNING: Could not parse training metrics for SEED {current_seed}: {e}")
        training_metrics = {}

    _ckpt_path = current_run_dir / "checkpoints" / "best.pt"
    _classif_dir = current_run_dir / "figures" / "classification"

    if not _ckpt_path.exists():
        print(f"ERROR: No checkpoint at {_ckpt_path} for SEED {current_seed}")
        print("Saving the training run to Drive anyway so the epoch history is not lost.")
        saved = _save_to_drive(current_run_dir, current_seed)
        if saved:
            saved_to_drive[current_seed] = saved
        continue

    # --eval-seed is FIXED; --tagger-checkpoint reuses one frozen HR tagger.
    eval_cmd = [
        sys.executable, "classification_eval.py",
        "--checkpoint", str(_ckpt_path),
        "--data-dir", str(DATA_DIR),
        "--out-dir", str(_classif_dir),
        "--eval-seed", str(EVAL_SEED),
        "--tagger-checkpoint", str(TAGGER_CKPT),
        "--semd-topk", str(SEMD_TOPK),
        "--semd-omega-R", str(SEMD_OMEGA_R),
        "--semd-max-samples", str(SEMD_MAX_SAMPLES),
    ]
    print("\nEVAL RUN:", end=" ")
    if _stream(eval_cmd) != 0:
        print(f"\nERROR: Evaluation failed for SEED {current_seed}.")
        print("Saving the training run to Drive anyway so the checkpoint is not lost.")
        saved = _save_to_drive(current_run_dir, current_seed)
        if saved:
            saved_to_drive[current_seed] = saved
        continue

    try:
        eval_res = json.loads((_classif_dir / "classification_eval.json").read_text())
        p = eval_res["primary_fixed_hr_tagger"]
        print("\n===== EVALUATION HEADLINE ====")
        print(f"AUC  HR={p['auc']['hr']:.3f}  LR={p['auc']['lr']:.3f}  SR={p['auc']['sr']:.3f}")
        print(f"tagging efficiency (AUC_SR/AUC_HR) = {p['tagging_efficiency_sr_over_hr']*100:.1f}%")
        print(f"recovery (LR->HR gap closed)       = {p['recovery_fraction_lr_to_hr']*100:.1f}%")

        _semd = eval_res.get("semd")
        if _semd:
            print(f"SEMD(SR,HR)={_semd['sr_vs_hr']['mean']:.5g}  "
                  f"SEMD(LR,HR)={_semd['lr_vs_hr']['mean']:.5g}  "
                  f"SEMD recovery={_semd['semd_recovery']*100:.1f}%   "
                  f"(top-{_semd['params']['topk']} approx.)")

        all_run_results[current_seed] = {
            "training_metrics": training_metrics,
            "evaluation_metrics": p,
            "semd": _semd,
            "eval_seed": EVAL_SEED,
        }
    except Exception as e:
        print(f"WARNING: Could not parse evaluation metrics for SEED {current_seed}: {e}")

    # ---- Persist THIS seed before starting the next one ----
    saved = _save_to_drive(current_run_dir, current_seed)
    if saved:
        saved_to_drive[current_seed] = saved

    # Running summary written after every seed, so partial sweeps stay usable.
    try:
        (DRIVE_RUNS_ROOT / f"multiseed_{SCALE}x_summary.json").write_text(
            json.dumps({
                "scale": SCALE, "epochs": EPOCHS, "eval_seed": EVAL_SEED,
                "seeds_completed": sorted(all_run_results),
                "results": all_run_results,
                "drive_paths": {str(k): str(v) for k, v in saved_to_drive.items()},
            }, indent=2, default=float), encoding="utf-8")
    except Exception as e:
        print(f"[drive] WARNING: could not write summary: {type(e).__name__}: {e}")

    print(f"\n{'='*80}\nFinished + SAVED SEED = {current_seed}\n{'='*80}")

print("\nAll multi-seed runs completed!")
print("Saved to Drive:")
for s, pth in sorted(saved_to_drive.items()):
    print(f"  seed {s}: {pth}")


## Cell 8 — Evaluate tagging efficiency on the trained checkpoint

In [ ]:
# This cell is now redundant. Its functionality has been moved into Cell 7 for multi-seed execution.

## Cell 9 — Review the sweep (already saved to Drive)
Each seed was copied to Drive by Cell 7 the moment it finished, so nothing is copied here. This cell builds the per-seed comparison table and checks that HR AUC stayed fixed across seeds.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import json, shutil

# Every seed was already copied to Drive by Cell 7 as it finished, so nothing is
# copied here -- this cell only REVIEWS the sweep and builds the comparison table.
DRIVE_RUNS_ROOT = Path("/content/drive/MyDrive/multiscale_sr_runs")

rows = []
for seed in SEEDS_TO_RUN:
    run_name = f"colab_{SCALE}x_full_{EPOCHS}_seed_{seed}"
    hits = sorted(DRIVE_RUNS_ROOT.glob(f"*{SCALE}x*{run_name}"))
    if not hits:
        print(f"seed {seed}: NOT on Drive (training or the copy failed)")
        continue
    run_dir = hits[-1]
    jf = run_dir / "figures" / "classification" / "classification_eval.json"
    if not jf.exists():
        print(f"seed {seed}: run saved but no evaluation JSON")
        continue
    p_ = json.loads(jf.read_text())["primary_fixed_hr_tagger"]
    rows.append((seed, p_["auc"]["hr"], p_["auc"]["lr"], p_["auc"]["sr"],
                 p_["tagging_efficiency_sr_over_hr"], p_["recovery_fraction_lr_to_hr"]))

if rows:
    print(f"\nMulti-seed comparison — scale {SCALE}x, {EPOCHS} epochs, eval_seed={EVAL_SEED} (fixed)\n")
    print(f"{'seed':>6} {'HR AUC':>8} {'LR AUC':>8} {'SR AUC':>8} {'eff':>8} {'recovery':>9}")
    for s, hr, lr, sr, eff, rec in rows:
        print(f"{s:>6} {hr:>8.4f} {lr:>8.4f} {sr:>8.4f} {eff*100:>7.1f}% {rec*100:>8.1f}%")

    import statistics as st
    srs = [r[3] for r in rows]
    hrs = [r[1] for r in rows]
    if len(srs) > 1:
        print(f"\nSR AUC: mean {st.mean(srs):.4f} +/- {st.stdev(srs):.4f} (n={len(srs)})")
        hr_spread = max(hrs) - min(hrs)
        print(f"HR AUC spread across seeds: {hr_spread:.2e}")
        if hr_spread < 1e-6:
            print("  OK: the frozen tagger held the ruler fixed, so SR spread is model variance.")
        else:
            print("  WARNING: HR AUC moved between seeds. HR does not depend on the generator,")
            print("  so this means a seed leaked into the evaluation -- investigate before")
            print("  interpreting the SR spread as model instability.")
    print("\nReport mean +/- std with the seed set and eval_seed attached. Never quote a single seed.")

# Zip the whole sweep for one download (optional).
try:
    zip_base = f"/content/multiseed_{SCALE}x_{EPOCHS}ep"
    shutil.make_archive(zip_base, "zip", root_dir=str(DRIVE_RUNS_ROOT))
    print("\nArchive:", zip_base + ".zip")
    from google.colab import files
    files.download(zip_base + ".zip")
except Exception as e:
    print("(download unavailable:", type(e).__name__, "— everything is already on Drive)")


## Cell 9a — Retro-evaluate saved checkpoints to ADD SEMD (eval only, no retraining)

Runs already in Drive were evaluated **before the SEMD code existed**, so their
`classification_eval.json` has no `semd` block and Cell 9b will (correctly) skip
them. This cell re-runs *evaluation only* on checkpoints already saved — no
training, so it costs eval time, not GPU-hours on the generator.

Two filters matter and both default to the safe choice:

- **`RETRO_SCALE`** — evaluate one scale at a time. Tagging difficulty depends
  strongly on scale, so correlating 16x/32x/64x together measures "which scale",
  not "which seed".
- **`RETRO_ONLY_EVAL_SEEDED`** — skip runs predating the Phase A eval-seed fix.
  In those, the tagger split moved with the training seed, so generator variance
  and evaluation noise are not separable.

Set `RETRO_FORCE = True` to redo runs that already have SEMD.

In [ ]:
import json, subprocess, sys
from pathlib import Path

RETRO_SCALE            = SCALE   # evaluate one scale at a time
RETRO_ONLY_EVAL_SEEDED = True    # skip pre-Phase-A runs (no fixed ruler)
RETRO_FORCE            = False   # re-do runs that already carry SEMD

_candidates = []
for _ckpt in sorted(DRIVE_RUNS_ROOT.glob("*/checkpoints/best.pt")):
    _run = _ckpt.parent.parent
    if f"_{RETRO_SCALE}x_" not in _run.name:
        continue
    _js = _run / "figures" / "classification" / "classification_eval.json"
    _has_semd, _eval_seed = False, None
    if _js.exists():
        try:
            _d = json.loads(_js.read_text())
            _has_semd = bool((_d.get("semd") or {}).get("params"))
            _eval_seed = _d.get("eval_seed")
        except Exception:
            pass
    if _has_semd and not RETRO_FORCE:
        print(f"[skip] {_run.name}: already has SEMD")
        continue
    if RETRO_ONLY_EVAL_SEEDED and _js.exists() and _eval_seed is None:
        print(f"[skip] {_run.name}: predates the eval-seed fix (ruler not fixed)")
        continue
    _candidates.append((_run, _ckpt))

print(f"\n{len(_candidates)} checkpoint(s) to retro-evaluate at {RETRO_SCALE}x\n")

for _run, _ckpt in _candidates:
    print(f"{'='*80}\n{_run.name}\n{'='*80}")
    _out = _run / "figures" / "classification"
    _out.mkdir(parents=True, exist_ok=True)
    _rc = subprocess.run([
        sys.executable, "classification_eval.py",
        "--checkpoint", str(_ckpt),
        "--data-dir", str(DATA_DIR),
        "--out-dir", str(_out),
        "--eval-seed", str(EVAL_SEED),
        "--tagger-checkpoint", str(TAGGER_CKPT),
        "--semd-topk", str(SEMD_TOPK),
        "--semd-omega-R", str(SEMD_OMEGA_R),
        "--semd-max-samples", str(SEMD_MAX_SAMPLES),
    ]).returncode
    if _rc != 0:
        print(f"FAILED: {_run.name} -- continuing with the rest\n")

print("\nRetro-evaluation done. Run Cell 9b for the verdict.")


## Cell 9b — Does SEMD see what the pixel-wise metrics cannot?

This is the decision point. Across the four seeds, tagging efficiency swings ~10%
while every logged physics metric stays flat to <1% — because all of them
(`energy_response`, `peak_ratio`, `nonzero_ratio`) are **invariant under pixel
permutation** and therefore blind to *where* energy sits. SEMD is not.

The cell below correlates **every** metric against tagging efficiency, SEMD
included and with no special treatment, and prints a verdict.

**Read the caveat it prints.** With n=4 the smallest attainable permutation
p-value is 0.083, so nothing here can reach significance — it ranks hypotheses,
it does not confirm one. If SEMD does *not* win, that is the finding: report it,
and do not sweep `topk`/`omega_R` hunting for a setting that reproduces the
ranking you already know.

In [ ]:
import subprocess, sys
from pathlib import Path

# Point this at the Drive directory holding the finished per-seed runs.
_corr_out = DRIVE_RUNS_ROOT / f"semd_correlation_{SCALE}x.md"

_rc = subprocess.run(
    [sys.executable, "semd_correlation.py",
     "--eval-dir", str(DRIVE_RUNS_ROOT),
     "--scale", str(SCALE),
     "--out", str(_corr_out),
     "--out-json", str(DRIVE_RUNS_ROOT / f"semd_correlation_{SCALE}x.json")],
    text=True,
)

if _rc.returncode != 0:
    print("\n[corr] failed. Most likely no classification_eval.json under",
          DRIVE_RUNS_ROOT, "-- finish the sweep in Cell 7 first.")
else:
    print(f"\n[corr] report saved to Drive: {_corr_out}")


## Cell 10 — After the sweep

**Results are on Drive**, one directory per seed under `MyDrive/multiscale_sr_runs/`, plus:
- `taggers/hr_tagger_<scale>x.pt` — the frozen tagger. **Keep this file.** Every future result must be measured against the same object, or the numbers are not comparable.
- `multiseed_<scale>x_summary.json` — rewritten after every seed, so a partial sweep is still usable.

**Resume a broken sweep:** just re-run Cell 7. Seeds already on Drive are skipped.

**Another scale:** set `SCALE` in Cell 6 and *Run all*. Each scale gets its own frozen tagger, since the tagger's train/test split depends on the materialized sample set.

**Reporting:** quote mean ± std over seeds with the seed set and `eval_seed` attached, never a single seed's number.